# Pydantic Tutorial

This notebook teaches the basics of Pydantic with short explanations followed by runnable examples.

## Example 1: Creating a Simple Model

A Pydantic model is created by extending `BaseModel`. Pydantic automatically validates the data types and can even convert compatible values.

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

user = User(name="John", age="25")

print(user)

## Example 2: Restricting Values with Literal

`Literal` allows a field to accept only specific values. This is useful when a field should be limited to a predefined list of options.

In [ ]:
from typing import Literal
from pydantic import BaseModel

class Ticket(BaseModel):
    priority: Literal["Low", "Medium", "High"]

ticket = Ticket(priority="High")

print(ticket)

In [ ]:
# This will not work
# ticket = Ticket(priority="Higher")

## Example 3: Building a Real-World Model

This example creates a regulatory intake record. The model contains multiple fields, each with validation rules using `Literal`.

In [ ]:
from typing import Literal
from pydantic import BaseModel, field_validator

class IntakeRecord(BaseModel):
    query_type: Literal[
        "complaint", "submission", "variation", "safety_signal",
        "label_update", "inspection", "general_enquiry"
    ]

    regulation_ref: Literal[
        "FDA_21CFR", "EMA_CTR", "ICH_E2A", "ICH_Q10",
        "CDSCO_NDC", "GxP_GMP", "GxP_GCP", "other"
    ]

    product_area: Literal[
        "oncology", "cardiovascular", "infectious_disease",
        "cmc", "clinical", "labelling", "general"
    ]

    urgency: Literal["routine", "standard", "urgent", "critical"]

    submitting_team: str

    @field_validator("submitting_team")
    @classmethod
    def team_must_not_be_empty(cls, v):
        if not v.strip():
            raise ValueError("submitting_team cannot be empty")
        return v.strip().title()

    @field_validator("submitting_team")
    @classmethod
    def team_not_a_person(cls, v):
        if len(v.split()) == 1 and v[0].isupper() and v.isalpha():
            raise ValueError(
                "submitting_team must be a team or function."
            )
        return v

## Example 4: Creating a Valid Object

A dictionary containing valid data is passed to the model. Pydantic validates the input and creates a strongly typed object.

In [ ]:
valid_data = {
    "query_type": "safety_signal",
    "regulation_ref": "ICH_E2A",
    "product_area": "clinical",
    "urgency": "critical",
    "submitting_team": "PV Team"
}

record = IntakeRecord(**valid_data)

print(record)
print(record.model_dump())

In [ ]:
invalid_data = {
    "query_type": "safety_signal",
    "regulation_ref": "ICH_E2A",
    "product_area": "clinical",
    "urgency": "critical",
    "submitting_team": "JOHN"
}

record = IntakeRecord(**invalid_data)

## Example 5: Handling Validation Errors

If invalid data is supplied, Pydantic raises a `ValidationError`. This example demonstrates how to capture and display detailed error messages.

In [11]:
from pydantic import ValidationError

try:
    bad_record = IntakeRecord(
        query_type="complaint",
        regulation_ref="WHO_GMP",
        product_area="clinical",
        urgency="critical",
        submitting_team="PV Team"
    )
except ValidationError as e:
    print("Validation failed:")

    for err in e.errors():
        print(
            f"Field: {err['loc'][0]} | Error: {err['msg']}"
        )

Validation failed:
Field: regulation_ref | Error: Input should be 'FDA_21CFR', 'EMA_CTR', 'ICH_E2A', 'ICH_Q10', 'CDSCO_NDC', 'GxP_GMP', 'GxP_GCP' or 'other'


## Summary

* `BaseModel` defines the structure of data.
* Type hints provide automatic validation.
* `Literal` restricts values.
* Validators implement custom business rules.
* `ValidationError` handles invalid data.
* `model_dump()` converts the model to a dictionary.